### Vector stores

> https://docs.langchain.com/oss/python/integrations/vectorstores#faiss

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

> https://docs.langchain.com/oss/python/integrations/vectorstores/faiss

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [6]:
# uv add langchain-community faiss-cpu
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. 벡터 차원 정의: 사용 중인 임베딩 모델이 생성하는 결과물의 길이(차원)를 계산합니다.
# 임베딩 모델(embeddings)은 사전에 정의되어 있어야 합니다.
embedding_dim = len(embeddings.embed_query("hello world"))

# 2. FAISS 인덱스 초기화: 벡터 간의 거리 계산 방식을 결정합니다.
# IndexFlatL2: 유클리드 거리($L2$ distance)를 계산하는 가장 정확한 완전 탐색(Brute-force) 방식입니다.
index = faiss.IndexFlatL2(embedding_dim)

# 3. LangChain FAISS 객체 조립: 검색 엔진(index)과 원본 데이터 저장소(docstore)를 결합합니다.
vector_store = FAISS(
    embedding_function=embeddings,  # 텍스트를 벡터로 변환할 함수
    index=index,                   # 유사도 검색을 수행할 FAISS 인덱스
    docstore=InMemoryDocstore(),   # 실제 텍스트 내용과 메타데이터를 담을 메모리 저장소
    index_to_docstore_id={},       # 인덱스 번호와 저장소 ID 간의 매핑 테이블(초기화 시 빈 값)
)

# 4. 데이터 준비: 검색 대상이 될 문서 객체들을 생성합니다.
documents = [
    Document(page_content="컴퓨터 공학은 하드웨어와 소프트웨어를 연구하는 학문입니다.", metadata={"source": "edu"}),
    Document(page_content="인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다.", metadata={"source": "tech"}),
    Document(page_content="고양이는 귀여운 동물이며 많은 사람들이 반려 동물로 키웁니다.", metadata={"source": "pets"}),
    Document(page_content="파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다.", metadata={"source": "tech"}),
]

# 5. 데이터 적재: 문서를 임베딩하여 벡터화한 뒤 FAISS 인덱스에 추가합니다.
# 내부적으로 임베딩 생성 -> 인덱스 저장 -> docstore 저장이 동시에 수행됩니다.
vector_store.add_documents(documents=documents)

# 6. 유사도 검색(Top-K): 질문과 의미적으로 가장 가까운 문서 k개를 추출합니다.
query = "머신러닝과 AI 기술에 대해 알려줘"
results = vector_store.similarity_search(query, k=2)

print(f"--- [검색 질의]: {query} ---")
for i, doc in enumerate(results):
    print(f"결과 {i+1}: {doc.page_content} (출처: {doc.metadata['source']})")

# 7. 점수 포함 검색: 거리 값(Distance)을 포함하여 검색 결과의 신뢰도를 확인합니다.
# IndexFlatL2를 사용하므로 점수(Score)는 거리를 의미하며, 0에 가까울수록 유사도가 높습니다.
results_with_score = vector_store.similarity_search_with_score(query, k=4)

print("\n--- [점수 포함 검색 결과] ---")
for doc, score in results_with_score:
    print(f"거리(Score): {score:.4f} | 내용: {doc.page_content}")

--- [검색 질의]: 머신러닝과 AI 기술에 대해 알려줘 ---
결과 1: 인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다. (출처: tech)
결과 2: 파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다. (출처: tech)

--- [점수 포함 검색 결과] ---
거리(Score): 0.5265 | 내용: 인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다.
거리(Score): 0.6992 | 내용: 파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다.
거리(Score): 0.7824 | 내용: 컴퓨터 공학은 하드웨어와 소프트웨어를 연구하는 학문입니다.
거리(Score): 0.8486 | 내용: 고양이는 귀여운 동물이며 많은 사람들이 반려 동물로 키웁니다.


---

In [7]:
# uv add langchain-community pdfplumber
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

In [8]:
# uv add langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

In [9]:
texts[0].page_content

'2026 년 테크노빌드 주식회사(TechnoBuild) 임직원\n통합 가이드북\n문서 번호: TB-HR-2026-001 (Rev 5.0)\n발행일: 2026 년 1 월 15 일\n적용 대상: 테크노빌드 전 임직원 (정규직, 계약직, 파견직 포함)\n주관 부서: 인사문화본부 (HR Culture Division)\n보안 등급: 대외비 (Internal Use Only)\n[목 차]\n1. 회사 개요 (Company Overview)\no 1.1 CEO 인사말\no 1.2 기업 미션 및 비전\no 1.3 핵심 가치 : T-SPIRIT\no 1.4 조직도 및 본부 소개\n2. 인사 및 평가 제도 (HR System)\no 2.1 직급 및 호칭 체계\no 2.2 승진 포인트 제도 (Tech-Point)'

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings,
    store,
    namespace=embeddings.model
)

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(texts, cached_embedder) # in-memory

In [ ]:
vector_store.save_local("./faiss_index") # 로컬 디스크에 저장하기

In [16]:
vector_store = None

In [17]:
vector_store

In [20]:
vector_store = FAISS.load_local(
    "./faiss_index",
    cached_embedder,
    allow_dangerous_deserialization=True
)

In [21]:
query = "임직원 통합 가이드북의 발행일은?"

In [22]:
vector_store.similarity_search(query, k=1)

[Document(id='9f331edb-c12e-4210-a6da-5720533c2901', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 0, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='2026 년 테크노빌드 주식회사(TechnoBuild) 임직원\n통합 가이드북\n문서 번호: TB-HR-2026-001 (Rev 5.0)\n발행일: 2026 년 1 월 15 일\n적용 대상: 테크노빌드 전 임직원 (정규직, 계약직, 파견직 포함)\n주관 부서: 인사문화본부 (HR Culture Division)\n보안 등급: 대외비 (Internal Use Only)\n[목 차]\n1. 회사 개요 (Company Overview)\no 1.1 CEO 인사말\no 1.2 기업 미션 및 비전\no 1.3 핵심 가치 : T-SPIRIT\no 1.4 조직도 및 본부 소개\n2. 인사 및 평가 제도 (HR System)\no 2.1 직급 및 호칭 체계\no 2.2 승진 포인트 제도 (Tech-Point)')]

In [23]:
query = "이번 주에 내가 결혼을 하는데 얼마를 받을 수 있을까?"

In [24]:
vector_store.similarity_search(query, k=1)

[Document(id='022721e5-cfa1-4685-931c-93dca45987c3', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 11, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='\uf0b7 소멸: 당해 연도 12 월 31 일까지 미사용 시 자동 소멸 (이월 불가).\n5.4 경조사 지원 기준 (Family Support Details)\n기쁨과 슬픔을 함께 나누는 테크노빌드의 경조 지원입니다.\n구분 대상 휴가 (일수) 경조금 (만원) 화환/조화\n결혼 본인 5 일 100 + 화환 지원\n자녀 1 일 50 + 화환 지원\n30 -\n형제/자매 1 일\n30 -\n회갑/칠순 본인/배우자 부모 1 일\n출산 본인 출산휴가 (90 일) 출산 축하금 50 과일 바구니\n배우자 10 일 (유급) 출산 축하금 50 과일 바구니\n-\n사망 본인/배우자 500 + 장례용품 3 단 조화 + 근조기\n부모/배우자 부모 5 일 100 + 장례용품 3 단 조화 + 근조기\n30\n조부모/외조부모 3 일 조화\n30\n형제/자매 3 일 조화')]

---

In [25]:
retriver = vector_store.as_retriever()

In [26]:
retriver

VectorStoreRetriever(tags=['FAISS', 'CacheBackedEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001542D1CCEF0>, search_kwargs={})

In [27]:
retriver.invoke(query)

[Document(id='022721e5-cfa1-4685-931c-93dca45987c3', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 11, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='\uf0b7 소멸: 당해 연도 12 월 31 일까지 미사용 시 자동 소멸 (이월 불가).\n5.4 경조사 지원 기준 (Family Support Details)\n기쁨과 슬픔을 함께 나누는 테크노빌드의 경조 지원입니다.\n구분 대상 휴가 (일수) 경조금 (만원) 화환/조화\n결혼 본인 5 일 100 + 화환 지원\n자녀 1 일 50 + 화환 지원\n30 -\n형제/자매 1 일\n30 -\n회갑/칠순 본인/배우자 부모 1 일\n출산 본인 출산휴가 (90 일) 출산 축하금 50 과일 바구니\n배우자 10 일 (유급) 출산 축하금 50 과일 바구니\n-\n사망 본인/배우자 500 + 장례용품 3 단 조화 + 근조기\n부모/배우자 부모 5 일 100 + 장례용품 3 단 조화 + 근조기\n30\n조부모/외조부모 3 일 조화\n30\n형제/자매 3 일 조화'),
 Document(id='0123136e-2c1f-4118-a1d4-bf1868de5b09', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employ